# RISNet Algorithm Comparison & Analysis

This notebook comprehensively compares all available beam sweep algorithms in RISNet:
- **Linear (Brute-Force)**: Exhaustive search
- **Coarse-Fine (Two-Phase)**: Center-out + refinement
- **Directional Exhaustive**: Multi-link optimization
- **ML-Guided**: ML predictor + exhaustive sweep
- **ML-Only**: Pure ML predictions

**Analysis includes:**
- SNR performance comparison
- **STEERING ANGLES for each algorithm** ← SHOWS WHICH ANGLES TO SEND TO RIS
- Number of angles tested
- Computational efficiency
- Phase patterns to send to RIS hardware

## Setup and Imports

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

risnet_path = Path('.').resolve().parent.parent / 'risnet'
sys.path.insert(0, str(risnet_path))

from core import RISNetwork, Physics
from controller.beamsweeping import SweepAlgorithmLoader

print("✓ All imports successful")
print(f"✓ RISNet path: {risnet_path}")

## 1. Create Test Network with Nodes

In [ ]:
net = RISNetwork()
net.add_ap('AP1', 0, 0, power_dBm=20)
net.add_ris('RIS1', 5, 0, N=16, bits=1)
net.add_ue('UE1', 3.5, 1.5)

print("Network Configuration:")
print("="*50)
net.list_nodes()
print("="*50)

ap = net.get('AP1')
ris = net.get('RIS1')
ue = net.get('UE1')

print(f"\nNode Details:")
print(f"  AP:  {ap.name:6} at ({ap.pos[0]:5.1f}, {ap.pos[1]:5.1f}, {ap.pos[2]:5.1f}) - TX Power: {ap.power_dBm} dBm")
print(f"  RIS: {ris.name:6} at ({ris.pos[0]:5.1f}, {ris.pos[1]:5.1f}, {ris.pos[2]:5.1f}) - Elements: {ris.N*ris.N}, Bits: {ris.bits}")
print(f"  UE:  {ue.name:6} at ({ue.pos[0]:5.1f}, {ue.pos[1]:5.1f}, {ue.pos[2]:5.1f})")

ap_ris_dist = np.linalg.norm(np.array(ris.pos[:2]) - np.array(ap.pos[:2]))
ris_ue_dist = np.linalg.norm(np.array(ue.pos[:2]) - np.array(ris.pos[:2]))
print(f"\nDistances:")
print(f"  AP → RIS: {ap_ris_dist:.2f} m")
print(f"  RIS → UE: {ris_ue_dist:.2f} m")

## 2. Test Baseline (No Beam Sweep)

In [ ]:
print("Baseline Connection Test (No Beam Sweep)")
print("="*60)

baseline_result = net.connect('AP1', 'RIS1', 'UE1')

print(f"Baseline Results:")
print(f"  SNR:          {baseline_result['snr_dB']:>10.2f} dB")
print(f"  Power:        {baseline_result['pwr_dBm']:>10.2f} dBm")
print(f"  Gain (linear):{baseline_result['gain_linear']:>10.2f}")
print(f"  Beam Angle:   {baseline_result['beam_angle']:>10.2f}°")
print("="*60)

## 3. Run All Sweep Algorithms

In [ ]:
algorithms_to_test = [
    ('linear', {'fov': 60, 'step': 10}),
    ('coarse-fine', {'fov': 60, 'step': 10, 'fine_span': 10, 'fine_res': 1}),
    ('directional-exhaustive', {'fov': 60, 'step': 10}),
    ('ml-guided', {'fov': 60, 'step': 10, 'fine_span': 10, 'fine_res': 1, 'ml_predictor': 'xgb', 'top_k': 3}),
    ('ml', {'fov': 60, 'step': 10, 'ml_predictor': 'xgb', 'top_k': 5}),
]

results_dict = {}
phase_patterns_dict = {}
print("Running sweep algorithms (using same RISNet functions as CLI)...\n")

for algo_name, params in algorithms_to_test:
    print(f"Testing: {algo_name.upper()}")
    print(f"  Parameters: {params}")
    
    try:
        algo = SweepAlgorithmLoader.get_algorithm(algo_name, net)
        result = algo.sweep('AP1', 'RIS1', 'UE1', **params)
        results_dict[algo_name] = result
        
        ris_obj = net.get('RIS1')
        
        if hasattr(ris_obj, 'quantized_phases') and ris_obj.quantized_phases is not None:
            phase_patterns_dict[algo_name] = {
                'quantized_phases': ris_obj.quantized_phases.copy(),
                'current_phases': ris_obj.current_phases.copy() if hasattr(ris_obj, 'current_phases') else None,
                'phase_states': ris_obj.phase_states.copy() if hasattr(ris_obj, 'phase_states') else None,
            }
        else:
            phase_patterns_dict[algo_name] = {'quantized_phases': np.zeros(ris_obj.N * ris_obj.N), 'current_phases': None, 'phase_states': None}
        
        if 'best_snr_fine' in result:
            best_snr = result['best_snr_fine']
            num_tested = len(result.get('local_coarse', [])) + len(result.get('local_fine', []))
        elif 'best_snr' in result:
            best_snr = result['best_snr']
            num_tested = result.get('num_angles_tested', len(result.get('snr', [])))
        else:
            best_snr = max(result.get('snr', [0]))
            num_tested = len(result.get('snr', []))
        
        print(f"  ✓ Best SNR: {best_snr:.2f} dB")
        print(f"  ✓ Angles tested: {num_tested}")
        print(f"  ✓ Phase pattern: {len(phase_patterns_dict[algo_name]['quantized_phases'])} elements")
        print(f"  ✓ Status: SUCCESS\n")
        
    except Exception as e:
        print(f"  ✗ Error: {str(e)[:80]}")
        print(f"  ✗ Status: FAILED\n")
        results_dict[algo_name] = None
        phase_patterns_dict[algo_name] = None

print("="*70)
print(f"Completed {len([r for r in results_dict.values() if r is not None])}/{len(algorithms_to_test)} algorithms")

## 4. Extract Results with STEERING ANGLES

In [ ]:
comparison_data = []

for algo_name, result in results_dict.items():
    if result is None:
        continue

    if 'best_snr_fine' in result:
        best_snr = result['best_snr_fine']
    elif 'best_snr' in result:
        best_snr = result['best_snr']
    else:
        best_snr = max(result.get('snr', [0]))

    if 'best_local_fine' in result:
        steering_angle_local = result['best_local_fine']
    elif 'best_local' in result:
        steering_angle_local = result['best_local']
    elif 'best_angle' in result:
        steering_angle_local = result['best_angle']
    else:
        steering_angle_local = None

    specular_angle = result.get('specular_angle', result.get('base_angle', 0))

    if steering_angle_local is not None:
        steering_angle_absolute = specular_angle + steering_angle_local
        steering_angle_absolute = steering_angle_absolute % 360
    else:
        steering_angle_absolute = specular_angle

    fov = 60
    angle_offset = (steering_angle_absolute - specular_angle + 180) % 360 - 180
    steering_angle_in_fov = abs(angle_offset) <= fov/2

    if 'num_angles_tested' in result:
        num_tested = result['num_angles_tested']
    elif 'local_coarse' in result and 'local_fine' in result:
        num_tested = len(result['local_coarse']) + len(result['local_fine'])
    else:
        num_tested = len(result.get('snr', []))

    snr_improvement = best_snr - baseline_result['snr_dB']

    comparison_data.append({
        'Algorithm': algo_name,
        'Best SNR (dB)': best_snr,
        'SNR Improvement (dB)': snr_improvement,
        'Specular Angle (°)': specular_angle,
        'Steering Angle Local (°)': steering_angle_local if steering_angle_local is not None else 0,
        'Steering Angle Absolute (°)': steering_angle_absolute,
        'In FOV': steering_angle_in_fov,
        'Angles Tested': num_tested,
        'Efficiency (dB/angle)': best_snr / num_tested if num_tested > 0 else 0
    })

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*150)
print("ALGORITHM COMPARISON RESULTS - WITH STEERING ANGLES")
print("="*150)

display_cols = ['Algorithm', 'Best SNR (dB)', 'SNR Improvement (dB)', 'Steering Angle Local (°)', 
                'Steering Angle Absolute (°)', 'Angles Tested', 'Efficiency (dB/angle)']
print(df_comparison[display_cols].to_string(index=False))

print("="*150)

## 5. STEERING ANGLES SUMMARY - WHAT TO SEND TO RIS

In [ ]:
print("\n" + "="*100)
print("STEERING ANGLES FOR EACH ALGORITHM - WHICH ANGLE TO SEND TO RIS?")
print("="*100)

for idx, row in df_comparison.iterrows():
    print(f"\n{row['Algorithm'].upper()}:")
    print(f"  ┌─ Specular Angle (reference):  {row['Specular Angle (°)']:7.2f}°")
    print(f"  ├─ Steering Angle (LOCAL):     {row['Steering Angle Local (°)']:+7.2f}° ← USE THIS")
    print(f"  ├─ Steering Angle (ABSOLUTE):  {row['Steering Angle Absolute (°)']:7.2f}°")
    print(f"  ├─ Best SNR:                   {row['Best SNR (dB)']:7.2f} dB")
    print(f"  ├─ SNR Improvement:            {row['SNR Improvement (dB)']:+7.2f} dB")
    print(f"  ├─ Angles Tested:              {row['Angles Tested']:7.0f}")
    print(f"  └─ Efficiency:                 {row['Efficiency (dB/angle)']:7.2f} dB/angle")

print("\n" + "="*100)
print("RECOMMENDATION: WHICH PHASE PATTERN TO SEND TO RIS?")
print("="*100)

best_idx = df_comparison['Best SNR (dB)'].idxmax()
best_row = df_comparison.loc[best_idx]

print(f"\n✓ BEST PERFORMANCE (Highest SNR): {best_row['Algorithm'].upper()}")
print(f"  ")
print(f"  Steering angle to send: {best_row['Steering Angle Local (°)']:+.2f}° (local/relative)")
print(f"  Or in absolute terms:   {best_row['Steering Angle Absolute (°)']:.2f}° (0-360° reference)")
print(f"  Expected SNR:           {best_row['Best SNR (dB)']:.2f} dB")
print(f"  Number of measurements: {best_row['Angles Tested']:.0f}")
print(f"  ")
print(f"  Phase pattern file to use:")
print(f"  → phase_pattern_{best_row['Algorithm']}_quantized.npy")

eff_idx = df_comparison['Efficiency (dB/angle)'].idxmax()
eff_row = df_comparison.loc[eff_idx]

print(f"\n✓ MOST EFFICIENT (Fewest Measurements): {eff_row['Algorithm'].upper()}")
print(f"  ")
print(f"  Steering angle to send: {eff_row['Steering Angle Local (°)']:+.2f}° (local/relative)")
print(f"  Or in absolute terms:   {eff_row['Steering Angle Absolute (°)']:.2f}° (0-360° reference)")
print(f"  Expected SNR:           {eff_row['Best SNR (dB)']:.2f} dB")
print(f"  Number of measurements: {eff_row['Angles Tested']:.0f}")
print(f"  ")
print(f"  Phase pattern file to use:")
print(f"  → phase_pattern_{eff_row['Algorithm']}_quantized.npy")

print(f"\n" + "="*100)

## 6. RIS Phase Patterns (Actual Hardware Patterns)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

plot_idx = 0
ris_obj = net.get('RIS1')
N = ris_obj.N

for algo_name in sorted(phase_patterns_dict.keys()):
    if phase_patterns_dict[algo_name] is None:
        continue

    ax = axes[plot_idx]
    plot_idx += 1

    quantized_phases = phase_patterns_dict[algo_name]['quantized_phases']
    phase_pattern_2d = quantized_phases.reshape(N, N)
    
    algo_row = df_comparison[df_comparison['Algorithm'] == algo_name]
    if not algo_row.empty:
        best_snr = algo_row['Best SNR (dB)'].values[0]
        best_angle = algo_row['Steering Angle Local (°)'].values[0]
    else:
        best_snr = 0
        best_angle = 0

    im = ax.imshow(phase_pattern_2d, cmap='hsv', origin='lower', vmin=0, vmax=2*np.pi, aspect='auto')
    ax.set_xlabel('X Element Index', fontsize=10, fontweight='bold')
    ax.set_ylabel('Y Element Index', fontsize=10, fontweight='bold')
    ax.set_title(f'{algo_name.upper()}\nAngle: {best_angle:+.1f}° | SNR: {best_snr:.2f} dB', 
                 fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.2, linestyle='--', color='white')
    plt.colorbar(im, ax=ax, label='Phase (rad)')

for idx in range(plot_idx, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

print("✓ RIS phase patterns visualized (ACTUAL phases sent to hardware)")

## 7. Save Results

In [ ]:
df_comparison.to_csv('algorithm_comparison_results.csv', index=False)

for algo_name in phase_patterns_dict.keys():
    if phase_patterns_dict[algo_name] is not None:
        quantized_phases = phase_patterns_dict[algo_name]['quantized_phases']
        np.save(f'phase_pattern_{algo_name}_quantized.npy', quantized_phases)
        np.savetxt(f'phase_pattern_{algo_name}_quantized.csv', quantized_phases, delimiter=',', fmt='%.6f')

print("✓ Results saved to files:")
print("  - algorithm_comparison_results.csv")
print("  - phase_pattern_*.npy (binary phase patterns)")
print("  - phase_pattern_*.csv (human-readable phases)")

print(f"\n✓ FINAL RECOMMENDATION:")
print(f"  Use algorithm: {df_comparison.loc[df_comparison['Best SNR (dB)'].idxmax(), 'Algorithm'].upper()}")
print(f"  Steering angle: {df_comparison.loc[df_comparison['Best SNR (dB)'].idxmax(), 'Steering Angle Local (°)']:+.2f}° (LOCAL)")
print(f"  Phase pattern file: phase_pattern_{df_comparison.loc[df_comparison['Best SNR (dB)'].idxmax(), 'Algorithm']}_quantized.npy")
print(f"  Expected SNR: {df_comparison['Best SNR (dB)'].max():.2f} dB")

In [ ]:
## 8. CLEAR STEERING ANGLE DISPLAY - What to Configure on RIS Hardware

# Import the steering angle extractor utility
import sys
sys.path.insert(0, str(risnet_path / 'examples'))
from steering_angle_extractor import print_steering_angle_clear, compare_steering_angles

print("\n" + "█"*100)
print("STEERING ANGLES - WHAT EXACTLY TO SEND TO RIS HARDWARE")
print("█"*100)

# Show steering angle for best algorithm
best_idx = df_comparison['Best SNR (dB)'].idxmax()
best_algo = df_comparison.loc[best_idx, 'Algorithm']
best_result = results_dict[best_algo]

print_steering_angle_clear(best_result, algo_name=best_algo)

# Compare all algorithms
print("\nComparison of all algorithms:")
compare_steering_angles(results_dict)

## 7b. STANDARDIZED CLI OUTPUT - Matching RISNet CLI Format

In [ ]:
# Import standardized CLI formatter
sys.path.insert(0, str(risnet_path / 'examples'))
from cli_output_formatter import print_sweep_summary, print_comparison_table, create_cli_record

print("\n" + "█"*100)
print("STANDARDIZED OUTPUT - MATCHING RISNET CLI FORMAT")
print("█"*100)

# Show CLI-format output for best algorithm
best_idx = df_comparison['Best SNR (dB)'].idxmax()
best_algo = df_comparison.loc[best_idx, 'Algorithm']
best_result = results_dict[best_algo]

print(f"\nAlgorithm: {best_algo.upper()}")
print_sweep_summary(best_result, algo_name=best_algo)

# Show comparison table in CLI format
print("\nAll Algorithms Comparison (CLI Format):")
print_comparison_table(results_dict)

# Create CLI records for storage
print("CLI Records created:")
for algo_name in sorted(results_dict.keys()):
    if results_dict[algo_name] is not None:
        record = create_cli_record('AP1', 'RIS1', 'UE1', results_dict[algo_name], algo_name)
        print(f"  ✓ {algo_name}: SNR={record['snr_dB']:.2f}dB, Angle={record['beam_angle_local']:+.2f}°")